# MINI Cells Experiment 013 — Random-Depth Ablation

A complete 2×2×2 matched-seed ablation of the three Experiment 011 ingredients: random recurrent depth, step-embedding initialization scale, and residual stability loss. Runs all eight cells for both 1D and 2D MiniCells.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
REF = os.environ.get('MINICELLS_REF', 'main')
os.chdir('/kaggle/working')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
os.chdir(ROOT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
import torch
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': torch.cuda.device_count(), 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.is_available(), 'CUDA is required'
if torch.cuda.device_count() < 2:
    print('Warning: Experiment 013 will run sequentially on one GPU; T4 x2 is recommended.')

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/research/01-foundations/test_language_depth_ablation.py',
    'tests/research/01-foundations/test_language_stabilization.py',
    'tests/research/01-foundations/test_language_2d.py',
    '-q',
], cwd=ROOT, check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/research/run_language_depth_ablation.py'], cwd=ROOT, check=True)

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
OUT = ROOT / 'results' / 'language-depth-ablation-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
summary = pd.read_csv(OUT / 'model-summary.csv')
effects = pd.read_csv(OUT / 'factorial-effects.csv')
contrasts = pd.read_csv(OUT / 'pure-contrasts.csv')
replication = pd.read_csv(OUT / 'replication-check.csv')
print(json.dumps(decision, indent=2))
display(summary)
display(effects[(effects.metric == 'final_ppl_2m') & (effects.order == 1)])
display(contrasts[contrasts.metric == 'final_ppl_2m'])
display(replication)

In [ ]:
for name in [
    'factorial-ppl.png',
    'factorial-training-cost.png',
    'factor-main-effects-ppl.png',
    'random-depth-isolation.png',
    '1d-depth-robustness.png',
    '2d-depth-robustness.png',
    'step-embedding-growth.png',
]:
    print(name)
    display(Image(filename=str(OUT / name)))

In [ ]:
# Review decision.json, replication-check.csv, and plots first. Then persist the curated result branch.
PUBLISH = False
if PUBLISH:
    subprocess.run([sys.executable, 'scripts/research/publish_experiment_013_results.py', '--push'], cwd=ROOT, check=True)